# Phase S1 Fast: Cooling Theory Validation (Quick Version)

## Reduced for speed: 8 runs total

## Goals (same as full version)
1. Validate BatchNorm as cooling mechanism
2. Measure activation variance → compute γ
3. Test heating effect (noise injection)
4. Controlled skip comparison

## Design: 8 runs (vs 64 in full version)
- 4 configs (core): None/LN/BN/Skip
- 2 D values: base_ch 32/96
- 1 seed
- 100 epochs (vs 200)

In [ ]:
# Setup
import os, json, math, time
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
from torch import linalg
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

if not os.path.exists('ThermoRG-NN'):
    !git clone https://github.com/xliu203/ThermoRG-NN.git
    %cd ThermoRG-NN
os.chdir('ThermoRG-NN')

In [ ]:
# FAST CONFIG: 4 core configs
CONFIGS = [
    ('None_NoSkip',  'none',       False),  # Baseline
    ('LN_NoSkip',    'layernorm',  False),  # LayerNorm
    ('BN_NoSkip',    'batchnorm',  False),  # BatchNorm (cooling)
    ('None_Skip',    'none',       True),   # Skip effect
]

# 2 D values for scaling
D_VALUES = [32, 96]  # ~0.2M and ~2M params
SEEDS = [42]  # 1 seed for speed
EPOCHS = 100  # 100 epochs (fast)
BATCH_SIZE = 128
LR = 0.01
MOMENTUM = 0.9

OUT_DIR = Path('./phase_s1_results')
OUT_DIR.mkdir(exist_ok=True)
print(f"Configs: {len(CONFIGS)}, D: {D_VALUES}, Runs: {len(CONFIGS)*len(D_VALUES)}")

In [ ]:
# Network with variance tracking
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, norm_type='none', use_skip=False, stride=1):
        super().__init__()
        self.conv = nn.Conv2d(in_ch, out_ch, 3, padding=1, stride=stride, bias=False)
        if norm_type == 'layernorm':
            self.norm = nn.LayerNorm([out_ch, 32, 32])
        elif norm_type == 'batchnorm':
            self.norm = nn.BatchNorm2d(out_ch)
        else:
            self.norm = nn.Identity()
        self.act = nn.GELU()
        if use_skip and in_ch == out_ch and stride == 1:
            self.skip = nn.Identity()
        elif use_skip:
            self.skip = nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False)
        else:
            self.skip = None
        self.var_in = self.var_out = self.count = 0

    def forward(self, x):
        self.var_in += x.float().var().item()
        out = self.norm(self.conv(x))
        out = self.act(out)
        self.var_out += out.float().var().item()
        self.count += 1
        if self.skip is not None:
            out = out + self.skip(x)
        return out
    
    def get_var_ratio(self):
        return (self.var_out / max(self.count,1)) / (self.var_in / max(self.count,1) + 1e-8)


class Net(nn.Module):
    def __init__(self, base_ch=64, norm_type='none', use_skip=False):
        super().__init__()
        channels = [3, base_ch, base_ch*2, base_ch*2]
        self.blocks = nn.ModuleList([
            ConvBlock(channels[i], channels[i+1], norm_type, use_skip)
            for i in range(3)
        ])
        self.pool = nn.AdaptiveAvgPool2d((1,1))
        self.fc = nn.Linear(channels[-1], 10)
    
    def forward(self, x):
        for b in self.blocks: x = b(x)
        return self.fc(self.pool(x).squeeze())
    
    def get_weights(self):
        return [m.weight.data for m in self.modules() if isinstance(m, nn.Conv2d)]
    
    def get_var_ratios(self):
        return [b.get_var_ratio() for b in self.blocks]

    def reset_vars(self):
        for b in self.blocks: b.var_in = b.var_out = b.count = 0

print("Network defined.")

In [ ]:
# J_topo computation
def compute_D_eff(W):
    if W.dim() == 4: W = W.view(W.size(0), -1)
    fro_sq = (W**2).sum().item()
    S = linalg.svd(W.to('cpu')).S
    return fro_sq / (S[0].item()**2 + 1e-12)

def compute_J_topo(weights):
    eta_vals, d_prev = [], 3.0
    for W in weights:
        if W.dim() == 4: W = W.view(W.size(0), -1)
        De = compute_D_eff(W)
        eta_vals.append(De / max(d_prev, 1e-8))
        d_prev = De
    return math.exp(-sum(abs(math.log(max(e,1e-12))) for e in eta_vals) / len(eta_vals))

def gamma_from_var_ratios(ratios, k=1.0):
    return -k * np.mean([math.log(max(r, 1e-8)) for r in ratios])

print("J_topo functions defined.")

In [ ]:
# Data
import torchvision.transforms as T
from torchvision.datasets import CIFAR10
from torch.utils.data import DataLoader

transform = T.Compose([
    T.RandomCrop(32, padding=4), T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize([0.4914,0.4822,0.4465],[0.2470,0.2435,0.2616])
])

train_ds = CIFAR10('./data', True, transform=transform, download=True)
val_ds = CIFAR10('./data', False, transform=T.ToTensor(), download=False)
train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, BATCH_SIZE, num_workers=2)
print(f"Train: {len(train_ds)}, Val: {len(val_ds)}")

In [ ]:
# Training
from scipy.optimize import curve_fit

def train_model(model, epochs, lr, momentum):
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=momentum, weight_decay=5e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    criterion = nn.CrossEntropyLoss()
    model = model.to(device)
    model.train()
    
    for ep in range(epochs):
        for X, y in train_loader:
            X, y = X.to(device), y.to(device)
            opt.zero_grad()
            loss = criterion(model(X), y)
            loss.backward()
            opt.step()
        scheduler.step()
    
    # Final eval
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for X, y in val_loader:
            X, y = X.to(device), y.to(device)
            val_loss += criterion(model(X), y).item() * X.size(0)
    return val_loss / len(val_ds)


def fit_scaling(Ds, losses):
    def law(D, a, b, E): return a * np.array(D)**(-b) + E
    try:
        popt, _ = curve_fit(law, np.array(Ds), np.array(losses), 
                          p0=[10, 0.5, 0.5], bounds=([0,0,0],[1000,5,10]), maxfev=5000)
        return {'alpha':popt[0], 'beta':popt[1], 'E':popt[2]}
    except: return {'alpha':None,'beta':None,'E':None}

print("Training functions defined.")

In [ ]:
# MAIN LOOP
from datetime import datetime

results = {'timestamp': datetime.now().isoformat(), 'configs': []}
t0 = time.time()

for cfg_name, norm, use_skip in CONFIGS:
    print(f"\n[{cfg_name}] norm={norm}, skip={use_skip}")
    cfg = {'name': cfg_name, 'norm': norm, 'skip': use_skip, 'D': {}}
    
    # J_topo at init
    m = Net(64, norm, use_skip)
    J = compute_J_topo(m.get_weights())
    cfg['J_topo_init'] = round(J, 4)
    print(f"  J_topo(init) = {J:.4f}")
    del m; torch.cuda.empty_cache()
    
    losses = []
    for ch in D_VALUES:
        print(f"  ch={ch}...", end=' ', flush=True)
        m = Net(ch, norm, use_skip)
        m.reset_vars()
        loss = train_model(m, EPOCHS, LR, MOMENTUM)
        ratios = m.get_var_ratios()
        g = gamma_from_var_ratios(ratios)
        losses.append(loss)
        cfg['D'][str(ch)] = {'loss': round(loss,4), 'gamma': round(g,4)}
        print(f"loss={loss:.4f}, γ={g:.4f}")
        del m; torch.cuda.empty_cache()
    
    # Scaling fit
    fit = fit_scaling(D_VALUES, losses)
    cfg['scaling'] = fit
    print(f"  Fit: β={fit.get('beta','-'):.4f}, E={fit.get('E','-'):.4f}")
    results['configs'].append(cfg)

print(f"\nTotal time: {(time.time()-t0)/60:.1f} min")

In [ ]:
# Save
with open(OUT_DIR / 'phase_s1_fast.json', 'w') as f:
    json.dump(results, f, indent=2)
print(f"Saved to {OUT_DIR/'phase_s1_fast.json'}")

In [ ]:
# Summary
print("\n" + "="*60)
print("SUMMARY")
print("="*60)

base = results['configs'][0]
base_beta = base['scaling'].get('beta') or 0.4

print(f"{'Config':<15} {'J':<8} {'γ':<8} {'β':<10} {'φ':<8} {'E':<10}")
print("-" * 60)
for c in results['configs']:
    s = c['scaling']
    beta = s.get('beta') or 0
    phi = beta / base_beta if base_beta > 0 else 0
    gamma = c['D'][str(D_VALUES[0])]['gamma']
    print(f"{c['name']:<15} {c['J_topo_init']:<8.4f} {gamma:<8.4f} {beta:<10.4f} {phi:<8.3f} {s.get('E','-'):<10.4f}")

print("\n" + "="*60)
print("COOLING ANALYSIS")
print("="*60)

def cfg(n): return results['configs'][n]

bn = cfg(2); ln = cfg(1); none = cfg(0); skip = cfg(3)
print(f"\n1. Normalization Cooling:")
print(f"   LN vs None: β={ln['scaling'].get('beta','-'):.4f} vs {none['scaling'].get('beta','-'):.4f}")
print(f"   BN vs None: β={bn['scaling'].get('beta','-'):.4f} vs {none['scaling'].get('beta','-'):.4f}")

print(f"\n2. Skip Cooling:")
print(f"   Skip vs None: β={skip['scaling'].get('beta','-'):.4f} vs {none['scaling'].get('beta','-'):.4f}")

print(f"\n3. Prediction:")
print(f"   φ_BN ≈ 0.66 (BatchNorm reduces β by 34%)")
print(f"   φ_skip ≈ 0.93-0.98 (skip weak cooling)")